# 03a — Tokenizer & Data Prep
**Run once. Takes ~5–10 minutes. Must complete before running 03b.**

### What this does
1. Loads all synthetic `.jsonl` files
2. Trains a shared SentencePiece BPE tokenizer on the combined corpus
3. Tokenizes and caches ALL datasets to disk as `.pt` tensors
4. Tokenizes FLORES dev + devtest and caches them too

### Output (saved to `notebooks/models/`)
- `shared_spm.model` — SentencePiece tokenizer
- `cache_beam_M1.pt`, `cache_beam_M10.pt`, ... — tokenized training data
- `cache_flores_dev.pt`, `cache_flores_devtest.pt` — tokenized eval data
- `vocab_info.json` — vocab size, special token IDs

### After this notebook
→ Open `03b_train.ipynb` to train the student model

In [ ]:
# Install if needed
# !pip install --quiet sentencepiece torch tqdm

In [ ]:
import json
import random
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import sentencepiece as spm
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

# ── PATHS (modify if on Kaggle/Lightning AI) ──────────────────────────────
ROOT        = Path("..")                         # MHD2 root
SYNTH_DIR   = ROOT / "data" / "synthetic"
FLORES_DEV  = ROOT / "data" / "flores" / "dev.jsonl"
FLORES_TEST = ROOT / "data" / "flores" / "devtest.jsonl"
MODEL_DIR   = ROOT / "notebooks" / "models"
CACHE_DIR   = MODEL_DIR / "cache"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPM_PREFIX  = str(MODEL_DIR / "shared_spm")
SPM_MODEL   = SPM_PREFIX + ".model"
VOCAB_INFO  = MODEL_DIR / "vocab_info.json"

# ── SETTINGS ──────────────────────────────────────────────────────────────
VOCAB_SIZE  = 32_000
MAX_LENGTH  = 128     # tokens (includes BOS/EOS)

# All dataset files to cache
DATASETS = {
    "beam_M1":   "eng_swh_beam_M1.jsonl",
    "beam_M10":  "eng_swh_beam_M10.jsonl",
    "top_p_M10": "eng_swh_top_p_M10.jsonl",
    "top_k_M10": "eng_swh_top_k_M10.jsonl",
    "dbs_M10":   "eng_swh_dbs_M10.jsonl",
    "mbr_M10":   "eng_swh_mbr_M10.jsonl",
}

print("✓ Paths configured")
print(f"  Synthetic dir: {SYNTH_DIR.resolve()}")
print(f"  Cache dir:     {CACHE_DIR.resolve()}")

In [ ]:
# ── STEP 1: Load all synthetic data for tokenizer training ────────────────

def load_jsonl(path: Path, expand_hypotheses: bool = True) -> Tuple[List[str], List[str]]:
    """
    Load synthetic .jsonl.
    expand_hypotheses=True: all M hypotheses become separate training pairs (paper setting).
    expand_hypotheses=False: only hyp_id==0 kept.
    """
    srcs, tgts = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            if not expand_hypotheses and ex.get("hyp_id", 0) != 0:
                continue
            s, t = ex["src"].strip(), ex["tgt"].strip()
            if s and t:
                srcs.append(s)
                tgts.append(t)
    return srcs, tgts


def load_flores(path: Path) -> Tuple[List[str], List[str]]:
    srcs, refs = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            srcs.append(ex["source"].strip())
            refs.append(ex["reference"].strip())
    return srcs, refs


# Collect ALL text for tokenizer training (use beam_M10 as it covers most vocab)
print("Loading data for tokenizer training...")
all_src, all_tgt = load_jsonl(SYNTH_DIR / "eng_swh_beam_M10.jsonl", expand_hypotheses=False)
print(f"  Tokenizer corpus: {len(all_src)} src + {len(all_tgt)} tgt = {len(all_src)+len(all_tgt)} lines")

In [ ]:
# ── STEP 2: Train SentencePiece tokenizer ─────────────────────────────────

if Path(SPM_MODEL).exists():
    print(f"✓ Tokenizer already exists: {SPM_MODEL} — skipping training")
else:
    CORPUS_FILE = str(CACHE_DIR / "spm_corpus.txt")
    print(f"Writing corpus to {CORPUS_FILE}...")
    with open(CORPUS_FILE, "w", encoding="utf-8") as f:
        for s in all_src:
            f.write(s + "\n")
        for t in all_tgt:
            f.write(t + "\n")

    print(f"Training SentencePiece BPE (vocab={VOCAB_SIZE})...")
    spm.SentencePieceTrainer.train(
        input=CORPUS_FILE,
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB_SIZE,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece="<pad>", unk_piece="<unk>",
        bos_piece="<s>", eos_piece="</s>",
        shuffle_input_sentence=True,
        num_threads=4,
    )
    print(f"✓ Tokenizer saved: {SPM_MODEL}")

# Load tokenizer
sp = spm.SentencePieceProcessor(model_file=SPM_MODEL)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()
ACTUAL_VOCAB = sp.get_piece_size()

print(f"\nVocab size: {ACTUAL_VOCAB} | PAD={PAD_ID} UNK={UNK_ID} BOS={BOS_ID} EOS={EOS_ID}")

# Save vocab info for downstream notebooks
vocab_info = {
    "vocab_size": ACTUAL_VOCAB,
    "pad_id": PAD_ID, "unk_id": UNK_ID,
    "bos_id": BOS_ID, "eos_id": EOS_ID,
    "max_length": MAX_LENGTH,
    "spm_model": SPM_MODEL,
}
with open(VOCAB_INFO, "w") as f:
    json.dump(vocab_info, f, indent=2)
print(f"✓ Vocab info saved: {VOCAB_INFO}")

In [ ]:
# ── STEP 3: Roundtrip + UNK sanity checks ─────────────────────────────────

print("Roundtrip check:")
for s in ["Hello, world!", "Habari yako?", all_src[0][:60], all_tgt[0][:60]]:
    ids = sp.encode(s, add_bos=False, add_eos=False)
    dec = sp.decode(ids)
    ok = "✅" if dec.strip().lower() == s.strip().lower() else "⚠️"
    print(f"  {ok}  '{s[:45]}'  →  {len(ids)} tokens  →  '{dec[:45]}'")

# UNK rate on M1 corpus
unk_hits = sum(1 for s in all_src[:1000] for tid in sp.encode(s) if tid == UNK_ID)
total_tok = sum(len(sp.encode(s)) for s in all_src[:1000])
print(f"\nUNK rate on 1k src sentences: {100*unk_hits/total_tok:.2f}% (want < 1%)")

In [ ]:
# ── STEP 4: Tokenize and cache ALL datasets ────────────────────────────────

def tokenize_and_cache(srcs: List[str], tgts: List[str], cache_path: Path, desc: str):
    """
    Tokenize parallel pairs, truncate to MAX_LENGTH, save as list of dicts.
    Each dict: {src: LongTensor, tgt: LongTensor}
    """
    if cache_path.exists():
        print(f"  ✓ Already cached: {cache_path.name} — skipping")
        return

    data = []
    skipped = 0
    for src, tgt in tqdm(zip(srcs, tgts), total=len(srcs), desc=desc):
        src_ids = [BOS_ID] + sp.encode(src, add_bos=False, add_eos=False) + [EOS_ID]
        tgt_ids = [BOS_ID] + sp.encode(tgt, add_bos=False, add_eos=False) + [EOS_ID]
        # Truncate
        src_ids = src_ids[:MAX_LENGTH]
        tgt_ids = tgt_ids[:MAX_LENGTH]
        # Skip pairs shorter than 3 tokens (BOS + 1 + EOS)
        if len(src_ids) < 3 or len(tgt_ids) < 3:
            skipped += 1
            continue
        data.append({
            "src": torch.tensor(src_ids, dtype=torch.long),
            "tgt": torch.tensor(tgt_ids, dtype=torch.long),
        })

    torch.save(data, cache_path)
    print(f"  ✓ Cached {len(data)} pairs → {cache_path.name}  (skipped {skipped} empty)")


# Cache all synthetic datasets
print("Tokenizing synthetic datasets...")
for key, fname in DATASETS.items():
    fpath = SYNTH_DIR / fname
    if not fpath.exists():
        print(f"  ⚠️  Not found: {fname} — skipping")
        continue
    cache_path = CACHE_DIR / f"cache_{key}.pt"
    srcs, tgts = load_jsonl(fpath, expand_hypotheses=True)
    tokenize_and_cache(srcs, tgts, cache_path, desc=key)

# Cache FLORES eval sets
print("\nTokenizing FLORES eval sets...")
for split_name, fpath in [("flores_dev", FLORES_DEV), ("flores_devtest", FLORES_TEST)]:
    cache_path = CACHE_DIR / f"cache_{split_name}.pt"
    srcs, refs = load_flores(fpath)
    # Save raw text separately (needed for sacreBLEU)
    raw_path = CACHE_DIR / f"raw_{split_name}.json"
    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump({"src": srcs, "ref": refs}, f, ensure_ascii=False, indent=2)
    tokenize_and_cache(srcs, refs, cache_path, desc=split_name)
    print(f"  ✓ Raw text saved: {raw_path.name}")

print("\n🎉 All caching complete!")

In [ ]:
# ── STEP 5: Final verification ────────────────────────────────────────────

print("Cache summary:")
for f in sorted(CACHE_DIR.glob("*.pt")):
    data = torch.load(f, weights_only=False)
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<35}  {len(data):>7} pairs  {size_mb:5.1f} MB")

print("\nVocab info:")
print(json.dumps(vocab_info, indent=2))

print("\n✅ 03a complete. You can now run 03b_train.ipynb")